# Question Answering with LangChain, OpenAI, and MultiQuery Retriever

This interactive workbook demonstrates example of Elasticsearch's [MultiQuery Retriever](https://api.python.langchain.com/en/latest/retrievers/langchain.retrievers.multi_query.MultiQueryRetriever.html) to generate similar queries for a given user input and apply all queries to retrieve a larger set of relevant documents from a vectorstore.

Before we begin, we first split the fictional workplace documents into passages with `langchain` and uses OpenAI to transform these passages into embeddings and then store these into Elasticsearch.

We will then ask a question, generate similar questions using langchain and OpenAI, retrieve relevant passages from the vector store, and use langchain and OpenAI again to provide a summary for the questions.

In [3]:
!python3 -m pip install -qU jq lark langchain langchain-elasticsearch langchain_openai tiktoken



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: C:\Users\priya\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [5]:
import sys

!{sys.executable} -m pip install -U jq lark langchain langchain-community langchain-elasticsearch langchain-openai tiktoken

  Using cached lark-1.3.1-py3-none-any.whl.metadata (1.8 kB)
  Using cached langchain_elasticsearch-1.0.0-py3-none-any.whl.metadata (8.2 kB)
  Using cached elasticsearch-8.19.3-py3-none-any.whl.metadata (9.1 kB)
  Using cached elastic_transport-8.17.1-py3-none-any.whl.metadata (3.8 kB)
Using cached lark-1.3.1-py3-none-any.whl (113 kB)
Using cached langchain_elasticsearch-1.0.0-py3-none-any.whl (54 kB)
Using cached elasticsearch-8.19.3-py3-none-any.whl (952 kB)
Using cached elastic_transport-8.17.1-py3-none-any.whl (64 kB)
   ---------------------------------------- 0.0/874.8 kB ? eta -:--:--
   ---------------------------------------- 874.8/874.8 kB 25.3 MB/s  0:00:00

   ----- ---------------------------------- 1/7 [lark]
   ----- ---------------------------------- 1/7 [lark]
   ----------------- ---------------------- 3/7 [elastic-transport]
   ----------------- ---------------------- 3/7 [elastic-transport]
  Attempting uninstall: tiktoken
   ----------------- ----------------------

  You can safely remove it manually.


In [7]:
import sys

!{sys.executable} -m pip install -U langchain langchain-classic langchain-community langchain-elasticsearch langchain-openai tiktoken jq lark

## Install packages and import modules

In [8]:

from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_elasticsearch import ElasticsearchStore
from langchain_openai.llms import OpenAI
from langchain.retrievers.multi_query import MultiQueryRetriever
from getpass import getpass

ModuleNotFoundError: No module named 'langchain.retrievers'

In [9]:
from langchain_openai import OpenAIEmbeddings, OpenAI
from langchain_elasticsearch import ElasticsearchStore
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
from getpass import getpass

## Connect to Elasticsearch

ℹ️ We're using an Elastic Cloud deployment of Elasticsearch for this notebook. If you don't have an Elastic Cloud deployment, sign up [here](https://cloud.elastic.co/registration?utm_source=github&utm_content=elasticsearch-labs-notebook) for a free trial. 

We'll use the **Cloud ID** to identify our deployment, because we are using Elastic Cloud deployment. To find the Cloud ID for your deployment, go to https://cloud.elastic.co/deployments and select your deployment.

We will use [ElasticsearchStore](https://api.python.langchain.com/en/latest/vectorstores/langchain.vectorstores.elasticsearch.ElasticsearchStore.html) to connect to our elastic cloud deployment, This would help create and index data easily.  We would also send list of documents that we created in the previous step

In [22]:
# https://www.elastic.co/search-labs/tutorials/install-elasticsearch/elastic-cloud#finding-your-cloud-id
ELASTIC_CLOUD_ID = getpass("Elastic Cloud ID: ")

# https://www.elastic.co/search-labs/tutorials/install-elasticsearch/elastic-cloud#creating-an-api-key
ELASTIC_API_KEY = getpass("Elastic Api Key: ")

# https://platform.openai.com/api-keys
OPENAI_API_KEY = getpass("OpenAI API key: ")

embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)
INDEX_NAME = "workplace-docs-multiquery"
vectorstore = ElasticsearchStore(
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
    index_name=INDEX_NAME, #give it a meaningful name,
    embedding=embeddings,
)

## Indexing Data into Elasticsearch
Let's download the sample dataset and deserialize the document.

In [13]:
from urllib.request import urlopen
import json

url = "https://raw.githubusercontent.com/elastic/elasticsearch-labs/main/example-apps/chatbot-rag-app/data/data.json"

response = urlopen(url)
data = json.load(response)

with open("temp.json", "w") as json_file:
    json.dump(data, json_file)

### Split Documents into Passages

We’ll chunk documents into passages in order to improve the retrieval specificity and to ensure that we can provide multiple passages within the context window of the final question answering prompt.

Here we are chunking documents into 800 token passages with an overlap of 400 tokens.

Here we are using a simple splitter but Langchain offers more advanced splitters to reduce the chance of context being lost.

In [16]:
from langchain_community.document_loaders import JSONLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


def metadata_func(record: dict, metadata: dict) -> dict:
    #Populate the metadata dictionary with keys name, summary, url, category, and updated_at.
    metadata["name"] = record.get("name")
    metadata["summary"] = record.get("summary")
    metadata["url"] = record.get("url")
    metadata["category"] = record.get("category")
    metadata["created_on"] = record.get("created_on")
    metadata["updated_at"] = record.get("updated_at")

    return metadata


# For more loaders https://python.langchain.com/docs/modules/data_connection/document_loaders/
# And 3rd party loaders https://python.langchain.com/docs/modules/data_connection/document_loaders/#third-party-loaders
loader = JSONLoader(
    file_path="temp.json",
    jq_schema=".[]",
    content_key="content",
    metadata_func=metadata_func,
)

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=800, chunk_overlap=400 #define chunk size and chunk overlap
)
docs = loader.load_and_split(text_splitter=text_splitter)

In [20]:
from openai import OpenAI
from getpass import getpass

OPENAI_API_KEY = getpass("OpenAI API key: ").strip()

client = OpenAI(api_key=OPENAI_API_KEY)

response = client.embeddings.create(
    model="text-embedding-3-small",
    input="test sentence"
)

print("OpenAI key works")
print(len(response.data[0].embedding))

OpenAI key works
1536


### Bulk Import Passages

Now that we have split each document into the chunk size of 800, we will now index data to elasticsearch using [ElasticsearchStore.from_documents](https://api.python.langchain.com/en/latest/vectorstores/langchain.vectorstores.elasticsearch.ElasticsearchStore.html#langchain.vectorstores.elasticsearch.ElasticsearchStore.from_documents).

We will use Cloud ID, Password and Index name values set in the `Create cloud deployment` step.

In [26]:
from langchain_openai import ChatOpenAI

In [27]:
documents = vectorstore.from_documents(
    docs,
    embeddings,
    index_name="workplace-docs-multiquery",
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
)

#llm = OpenAI(temperature=0.6, openai_api_key=OPENAI_API_KEY)
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.6,
    api_key=OPENAI_API_KEY,
)

retriever = MultiQueryRetriever.from_llm(vectorstore.as_retriever(), llm)

# Question Answering with MultiQuery Retriever

Now that we have the passages stored in Elasticsearch, we can now ask a question to get the relevant passages.

In [29]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.prompts import format_document

import logging

logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

LLM_CONTEXT_PROMPT = ChatPromptTemplate.from_template(
    """You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Be as verbose and educational in your response as possible. 
    
    context: {context}
    Question: "{question}"
    Answer:
    """
)

LLM_DOCUMENT_PROMPT = PromptTemplate.from_template(
    """
---
SOURCE: {name}
{page_content}
---
"""
)


def _combine_documents(
    docs, document_prompt=LLM_DOCUMENT_PROMPT, document_separator="\n\n"
):
    doc_strings = [format_document(doc, document_prompt) for doc in docs]
    return document_separator.join(doc_strings)


_context = RunnableParallel(
    context=retriever | _combine_documents,
    question=RunnablePassthrough(),
)

chain = _context | LLM_CONTEXT_PROMPT | llm

ans = chain.invoke("what is the nasa sales team?")

print("---- Answer ----")
print(ans)

---- Answer ----
content='The NASA sales team refers to the sales organization that operates within the North America and South America region. This region is specifically managed by two Area Vice-Presidents: Laura Martinez oversees the North America segment, while Gary Johnson is responsible for the South America segment. The NASA sales team is tasked with addressing the unique market dynamics and customer needs across these two diverse regions.\n\nIn essence, the NASA sales team plays a crucial role in identifying and pursuing new business opportunities, nurturing existing client relationships, and ensuring customer satisfaction. They work collaboratively with other departments, such as marketing and product development, to deliver high-quality products and services tailored to their clients in both North and South America. This collaboration is vital for adapting to the distinct cultural and economic conditions present in each market, allowing the sales team to effectively engage wi

**Generate at least two new iteratioins of the previous cells - Be creative.** Did you master Multi-
Query Retriever concepts through this lab?

In [30]:
customer_support_question = """
A customer says: "My order has not arrived and I don't know who to contact."
Using the retrieved company documents, explain what support steps should be followed.
If the documents do not contain enough information, say clearly what is missing.
"""

ans_customer_support = chain.invoke(customer_support_question)

print("---- Customer Support Use Case Answer ----")
print(ans_customer_support.content if hasattr(ans_customer_support, "content") else ans_customer_support)

---- Customer Support Use Case Answer ----
The retrieved company documents do not provide specific information regarding customer support processes or contact procedures for inquiries about order statuses. Therefore, I cannot provide detailed support steps based on the available context.

However, generally speaking, if a customer is inquiring about an order that has not arrived, it is advisable for them to take the following steps:

1. **Check Order Confirmation**: The customer should first check their email for an order confirmation message, which typically includes tracking information or contact details for customer support.

2. **Visit the Company Website**: If available, the customer can visit the company’s website and look for a "Contact Us" section. This often contains phone numbers, email addresses, or live chat options.

3. **Contact Customer Service**: The customer should reach out to the customer service department directly through the provided contact methods. They should 

In [31]:
process_owner_question = """
A process owner wants to prepare a technical concept for a new automation project.
Using the retrieved company documents, explain what information should be included:
problem statement, architecture, data sources, timeline, budget, risks, and compliance.
If the documents do not contain enough information, mention the missing parts.
"""

ans_process_owner = chain.invoke(process_owner_question)

print("---- Process Owner Chatbot Use Case Answer ----")
print(ans_process_owner.content if hasattr(ans_process_owner, "content") else ans_process_owner)

---- Process Owner Chatbot Use Case Answer ----
To prepare a technical concept for a new automation project, a process owner should include several critical components in their documentation. Based on the retrieved company documents, here’s a detailed overview of the necessary elements, along with insights into what information may be missing.

1. **Problem Statement**: This section should clearly articulate the specific problem or challenge that the automation project aims to address. It should explain why automation is necessary and the expected benefits, such as increased efficiency, reduced errors, or cost savings. The documents retrieved do not provide specific examples of problem statements, so the owner will need to craft this based on the context of the project.

2. **Architecture**: In this part, the technical concept should outline the proposed architecture of the automation system. This includes the overall design, key components, and how they will interact. It may involve d